In [3]:
import duckdb
import pandas as pd
import requests
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("API_KEY")
API_KEY is not None

True

In [133]:
#loading raw file
df=pd.read_csv('data/revenues_per_day_fm.csv')



In [134]:
#file size check
df.shape

(337818, 6)

In [135]:
#first 5 rows
df.head()

,id,date,title,revenue,theaters,distributor
0,8b19ad43-3a7e-b14b-49e9-1f7a0eb1568e,2004-09-20,Sky Captain and the World of Tomorrow,925482,3170.0,Paramount Pictures
1,481fc700-fcdd-1919-c53c-09fcd423a596,2004-09-20,Resident Evil: Apocalypse,643680,3284.0,Screen Gems
2,06719cc2-c05e-8c0b-56b6-b8b13e056509,2004-09-20,Mr 3000,425375,2736.0,Walt Disney Studios Motion Pictures
3,3be8e4a8-2716-00be-80be-eb342ba3cbe7,2004-09-20,Wimbledon,416970,2034.0,Universal Pictures
4,c5f26b59-6a64-ae8e-8f58-7cefb09eadfe,2004-09-20,Cellular,412000,2749.0,New Line Cinema


In [136]:
df.describe()

,revenue,theaters
count,3.378180e+05,337657.000000
mean,6.071642e+05,848.611597
std,2.323203e+06,1137.442240
min,0.000000e+00,1.000000
25%,6.592000e+03,45.000000
50%,4.077500e+04,226.000000
75%,3.029205e+05,1402.000000
max,1.574616e+08,4931.000000


In [137]:
df.dtypes

id                 str
date               str
title              str
revenue          int64
theaters       float64
distributor        str
dtype: object

In [138]:
#how many nulls by columns
df.isna().sum()

#0.05% nulls in theaters column, and 0.07% of nulls in distributor column.

id               0
date             0
title            0
revenue          0
theaters       161
distributor    225
dtype: int64

In [139]:
#fully duplicated rows
df.duplicated().sum()

np.int64(0)

In [140]:
#errors in data type change
df["date"].isna().sum()    


np.int64(0)

In [141]:
#connecting duckdb and creating table for sql
con = duckdb.connect('data/revenues_per_day_fm.csv')
con.sql("CREATE OR REPLACE TABLE raw_revenues AS SELECT * FROM read_csv_auto('data/revenues_per_day_fm.csv')")

In [142]:
con.sql("DESCRIBE raw_revenues")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ title       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ revenue     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ theaters    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ distributor │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [143]:
#how many rows (should be the same as in pandas)
con.sql("SELECT COUNT(*) FROM raw_revenues")


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       337818 │
└──────────────┘

In [144]:
con.sql("SELECT MIN(date), MAX(date) FROM raw_revenues")

┌────────────┬────────────┐
│ min(date)  │ max(date)  │
│    date    │    date    │
├────────────┼────────────┤
│ 2000-01-01 │ 2023-03-06 │
└────────────┴────────────┘

In [145]:
#how many unique titles
con.sql("SELECT DISTINCT(title) FROM raw_revenues")

┌──────────────────────────────────────────────────────────┐
│                          title                           │
│                         varchar                          │
├──────────────────────────────────────────────────────────┤
│ San Andreas                                              │
│ Magic Mike XXL                                           │
│ Four Brothers                                            │
│ Broken Flowers                                           │
│ Fantastic Four                                           │
│ War of the Worlds                                        │
│ Deuce Bigalow: European Gigolo                           │
│ The Longest Yard                                         │
│ Cinderella Man                                           │
│ 8 Mile                                                   │
│   ·                                                      │
│   ·                                                      │
│   ·                   

In [146]:
con.sql("SELECT COUNT(id) FROM raw_revenues GROUP BY id HAVING COUNT(id)>1")

┌───────────┐
│ count(id) │
│   int64   │
└───────────┘
   0 rows  

In [147]:
con.sql("SELECT title, date, COUNT(*) FROM raw_revenues GROUP BY title, date HAVING COUNT(*)>1")

┌─────────┬──────┬──────────────┐
│  title  │ date │ count_star() │
│ varchar │ date │    int64     │
└─────────┴──────┴──────────────┘
             0 rows            

In [148]:
con.sql("SELECT id, title, revenue FROM raw_revenues WHERE revenue <= 0 ORDER BY revenue")

┌──────────────────────────────────────┬──────────────────────────────────────────────────────┬─────────┐
│                  id                  │                        title                         │ revenue │
│               varchar                │                       varchar                        │  int64  │
├──────────────────────────────────────┼──────────────────────────────────────────────────────┼─────────┤
│ 4cd24fa3-e62d-ded1-cd24-035a3afa4b01 │ White Snake                                          │       0 │
│ 338ad2c9-438f-1b97-c970-e452ace6e26d │ 16 Bars                                              │       0 │
│ b54493e6-4d31-1f1b-45ea-e7855d9e3c60 │ The Times of Bill Cunningham                         │       0 │
│ e8396d5b-7ae8-5559-ae54-c8445c696014 │ Incitement                                           │       0 │
│ 08af2a23-92ea-a3e0-0a26-4b759cc301c4 │ Counter Column                                       │       0 │
│ 89bc5398-9498-e752-e058-bbb0fa09bcdd │ The G

In [149]:
con.sql("SELECT id, date, title, revenue FROM raw_revenues ORDER BY revenue DESC LIMIT 5")
#Avengers: Endgame earned 157mln$ which is true, so revenue shows in usd.
#This also means that there's 63 rows of movies earning 0 usd in the record.

┌──────────────────────────────────────┬────────────┬────────────────────────────────────────────┬───────────┐
│                  id                  │    date    │                   title                    │  revenue  │
│               varchar                │    date    │                  varchar                   │   int64   │
├──────────────────────────────────────┼────────────┼────────────────────────────────────────────┼───────────┤
│ c5477391-de7d-0cb3-4738-bb9e971d6b53 │ 2019-04-26 │ Avengers: Endgame                          │ 157461641 │
│ e5f9c957-c088-7070-b32c-4e949c0c7d3d │ 2021-12-17 │ Spider-Man: No Way Home                    │ 121850000 │
│ 03ec9798-2fb1-b3e9-b711-3f0a6932f82f │ 2015-12-18 │ Star Wars: Episode VII - The Force Awakens │ 119119282 │
│ c88d93b6-2571-0f47-6e89-e40b0835361a │ 2019-04-27 │ Avengers: Endgame                          │ 109264122 │
│ b9a734bf-fe3f-42cc-404e-8cade0cfc075 │ 2018-04-27 │ Avengers: Infinity War                     │ 106334939 │
└

In [150]:
con.sql("SELECT COUNT(DISTINCT distributor) AS cdis, title FROM raw_revenues GROUP BY title HAVING cdis>1")

┌───────┬────────────────────────────┐
│ cdis  │           title            │
│ int64 │          varchar           │
├───────┼────────────────────────────┤
│     2 │ Undertow                   │
│     2 │ Breathe                    │
│     2 │ Shithouse                  │
│     2 │ The Road                   │
│     2 │ Thank You for Your Service │
│     2 │ Deliver Us from Evil       │
│     2 │ The Promise                │
│     2 │ Infinity Pool              │
│     2 │ Iris                       │
│     2 │ Pinocchio                  │
│     · │     ·                      │
│     · │     ·                      │
│     · │     ·                      │
│     2 │ The Way Back               │
│     2 │ Spiral                     │
│     2 │ Robin Hood                 │
│     2 │ Medieval                   │
│     2 │ Death at a Funeral         │
│     3 │ Last Call                  │
│     2 │ Camp                       │
│     2 │ Venom                      │
│     2 │ RRR2022 Re-rele

In [151]:
con.sql("SELECT distributor, COUNT(*) FROM raw_revenues GROUP BY distributor ORDER BY COUNT(*) ")

┌─────────────────────────────────────┬──────────────┐
│             distributor             │ count_star() │
│               varchar               │    int64     │
├─────────────────────────────────────┼──────────────┤
│ Fox Trail Productions               │            1 │
│ TLA Releasing                       │            1 │
│ Paladin                             │            1 │
│ Metrograph Pictures                 │            1 │
│ Archway Pictures                    │            1 │
│ PBS Distribution                    │            1 │
│ CFI Releasing                       │            1 │
│ FilmDemic                           │            1 │
│ Amplify                             │            1 │
│ Utopia                              │            1 │
│   ·                                 │            · │
│   ·                                 │            · │
│   ·                                 │            · │
│ The Weinstein Company               │        10120 │
│ Roadside

In [152]:
con.sql("SELECT title, MIN(date), MAX(date), MAX(date)-MIN(date) AS diff FROM raw_revenues GROUP BY title ORDER BY diff DESC")

┌─────────────────────────────────────────────────────────────────┬────────────┬────────────┬───────┐
│                              title                              │ min(date)  │ max(date)  │ diff  │
│                             varchar                             │    date    │    date    │ int64 │
├─────────────────────────────────────────────────────────────────┼────────────┼────────────┼───────┤
│ Charlie's Angels                                                │ 2000-11-03 │ 2020-01-16 │  7013 │
│ The Kid                                                         │ 2000-07-07 │ 2019-04-18 │  6859 │
│ Pinocchio                                                       │ 2002-12-25 │ 2021-03-04 │  6644 │
│ The Lost City                                                   │ 2006-04-28 │ 2022-06-23 │  5900 │
│ The Grudge                                                      │ 2004-10-22 │ 2020-02-27 │  5606 │
│ A Grin Without A Cat                                            │ 2002-05-01 │ 2

In [153]:
con.sql("SELECT id, title, date FROM raw_revenues WHERE title IN ('Charlie''s Angiels', 'The Kid', 'Pinocchio')")
#id is unique for title and day, not just for distinct movies.

┌──────────────────────────────────────┬───────────┬────────────┐
│                  id                  │   title   │    date    │
│               varchar                │  varchar  │    date    │
├──────────────────────────────────────┼───────────┼────────────┤
│ 3a733fdb-c80f-55e1-c90e-09a6c9e0ec07 │ The Kid   │ 2000-08-08 │
│ b57cfc95-88de-90fb-b972-f078ab08de22 │ The Kid   │ 2019-03-10 │
│ e8663642-f860-0a7a-f1eb-0e6bf3b617ac │ The Kid   │ 2000-07-30 │
│ 536fc148-d9ee-7d45-0113-f00fb8be4e5c │ The Kid   │ 2000-08-29 │
│ 2f3e9a24-acae-dbbb-0bbc-5433d91f9f21 │ The Kid   │ 2019-03-23 │
│ 121a805d-5757-2bf5-dc71-ed7107b5a9ba │ The Kid   │ 2019-04-12 │
│ b71484e3-5aba-a5ae-b97e-b1ea7c073315 │ Pinocchio │ 2003-01-02 │
│ 9a4938ba-a64d-b118-f777-27171bf4fa1a │ The Kid   │ 2019-03-28 │
│ dd403914-594d-bce2-4ed4-0704f9c57084 │ The Kid   │ 2000-08-06 │
│ 015423ca-6df5-8ea1-c908-73e3597edac7 │ The Kid   │ 2000-07-18 │
│                  ·                   │    ·      │     ·      │
│         

In [154]:
#some movies are remakes, and some are re-releases. In this CTE assumption is if there is no previous date (null), or date difference is larger than a year it means screening ended and we treat it as another movie. It's a hypothesis and we'll check it with OMDb.
con.sql("""
WITH lag_date AS(
    SELECT id, title, date, LAG(date) OVER (PARTITION BY title ORDER BY date) AS prev_date FROM raw_revenues
),
date_diff AS (
SELECT 
    id, 
    title, 
    date, 
    prev_date, 
    (date)-(prev_date) AS days_between, 
    CASE WHEN prev_date IS NULL THEN 1
        WHEN date_diff('day', prev_date, date) > 365 THEN 1
        ELSE 0
        END AS new_series
FROM lag_date),
series_number AS (
SELECT
    *,
    SUM(new_series) OVER (PARTITION BY title ORDER BY date) AS series_number
FROM date_diff
)
SELECT title, series_number, MIN(date), MAX(date), year(MIN(date)), COUNT(*) AS count FROM series_number GROUP BY title, series_number  ORDER BY title, series_number


""")

┌────────────────────────────────────────────┬───────────────┬────────────┬────────────┬───────────────────┬───────┐
│                   title                    │ series_number │ min(date)  │ max(date)  │ "year"(min(date)) │ count │
│                  varchar                   │    int128     │    date    │    date    │       int64       │ int64 │
├────────────────────────────────────────────┼───────────────┼────────────┼────────────┼───────────────────┼───────┤
│ '71                                        │             1 │ 2015-02-27 │ 2015-05-07 │              2015 │    70 │
│ '83                                        │             1 │ 2021-12-24 │ 2022-01-16 │              2021 │    14 │
│ '85: The Greatest Team in Football History │             1 │ 2018-01-29 │ 2018-02-07 │              2018 │     2 │
│ 10 Cloverfield Lane                        │             1 │ 2016-03-11 │ 2016-06-02 │              2016 │    84 │
│ 10 Years                                   │             1 │ 2

In [155]:
#connecting to database

In [10]:
resp = requests.get("https://www.omdbapi.com/",
                    params={"apikey": API_KEY, "t": "The Kid", "y":2019})
data = resp.json()
data

{'Title': 'The Kid',
 'Year': '2019',
 'Rated': 'R',
 'Released': '08 Mar 2019',
 'Runtime': '100 min',
 'Genre': 'Biography, Drama, Western',
 'Director': "Vincent D'Onofrio",
 'Writer': "Vincent D'Onofrio, Andrew Lanham",
 'Actors': 'Jake Schur, Leila George, Chris Pratt',
 'Plot': "The story of a young boy who witnesses Billy the Kid's encounter with Sheriff Pat Garrett.",
 'Language': 'English, Spanish',
 'Country': 'United States',
 'Awards': 'N/A',
 'Poster': 'https://m.media-amazon.com/images/M/MV5BNTI0NzQxOTMwOF5BMl5BanBnXkFtZTgwMjAxMzg0NzM@._V1_SX300.jpg',
 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '5.9/10'},
  {'Source': 'Rotten Tomatoes', 'Value': '43%'},
  {'Source': 'Metacritic', 'Value': '51/100'}],
 'Metascore': '51',
 'imdbRating': '5.9',
 'imdbVotes': '9,725',
 'imdbID': 'tt4975920',
 'Type': 'movie',
 'DVD': 'N/A',
 'BoxOffice': '$1,508,593',
 'Production': 'N/A',
 'Website': 'N/A',
 'Response': 'True'}

In [ ]:
#